# Bluebikes Data Pipeline
## How Did the Pandemic Reshape Boston's Bike-Share?

This notebook pulls the full Bluebikes trip data for 2019, 2022, and 2023 (about 9.97 million trips across 36 monthly CSV files), cleans it, and then rolls it up into a handful of summary datasets that the visualizations read from. We build one dataset per visualization so each chart only loads what it needs instead of re-aggregating millions of rows at display time.

## Imports

In [1]:
import pandas as pd
import os
import glob

## Data Source

Bluebikes publishes quarterly trip history data on their [system-data page](https://bluebikes.com/system-data). Each row in the raw files is one trip, meaning one rider on one bike going from station A to station B. We downloaded all 36 monthly CSV files across our three target years. The combined raw data comes out to roughly **9.97 million trip records**.

In [2]:
RAW_DIR = "/Users/gaurangdeka/BlueBikes/data/raw"
OUTPUT_DIR = "data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

YEARS = [2019, 2022, 2023]

## Schema Standardization

Bluebikes changed their data format partway through 2023, so we had to handle two different schemas. The **old schema** (used in 2019, 2022, and early 2023) has columns like `tripduration`, `starttime`, `birth year`, and `gender`. The **new schema** (used from late 2023 onward) renames most of those to `started_at`, `ended_at`, `member_casual`, and drops the demographic fields entirely. We map both into a single consistent schema before concatenating.

In [3]:
OLD_TO_STANDARD = {
    'tripduration': 'tripduration',
    'starttime': 'starttime',
    'stoptime': 'stoptime',
    'start station id': 'start_station_id',
    'start station name': 'start_station_name',
    'start station latitude': 'start_lat',
    'start station longitude': 'start_lng',
    'end station id': 'end_station_id',
    'end station name': 'end_station_name',
    'end station latitude': 'end_lat',
    'end station longitude': 'end_lng',
    'bikeid': 'bikeid',
    'usertype': 'usertype',
    'birth year': 'birth_year',
    'gender': 'gender',
    'postal code': 'postal_code',
}

NEW_TO_STANDARD = {
    'ride_id': 'ride_id',
    'rideable_type': 'rideable_type',
    'started_at': 'starttime',
    'ended_at': 'stoptime',
    'start_station_name': 'start_station_name',
    'start_station_id': 'start_station_id',
    'end_station_name': 'end_station_name',
    'end_station_id': 'end_station_id',
    'start_lat': 'start_lat',
    'start_lng': 'start_lng',
    'end_lat': 'end_lat',
    'end_lng': 'end_lng',
    'member_casual': 'usertype',
}

def standardize_df(df):
    cols = set(df.columns)
    if 'started_at' in cols or 'ride_id' in cols:
        df = df.rename(columns=NEW_TO_STANDARD)
        if 'usertype' in df.columns:
            df['usertype'] = df['usertype'].map({'member': 'Subscriber', 'casual': 'Customer'}).fillna(df['usertype'])
    else:
        df = df.rename(columns=OLD_TO_STANDARD)
    return df

## Loading Raw Data

We load all 36 monthly CSVs, standardize the column names, and parse datetime fields at the file level.

In [4]:
frames = []
for year in YEARS:
    pattern = os.path.join(RAW_DIR, f"{year}*-bluebikes-tripdata.csv")
    files = sorted(glob.glob(pattern))
    for f in files:
        df = pd.read_csv(f, dtype=str)
        df.columns = [c.strip().strip('"') for c in df.columns]
        df = standardize_df(df)
        df['starttime'] = pd.to_datetime(df['starttime'], errors='coerce')
        df['stoptime'] = pd.to_datetime(df['stoptime'], errors='coerce')
        frames.append(df)
        print(f"Loaded {os.path.basename(f)}: {len(df):,} rows")

raw = pd.concat(frames, ignore_index=True)
print(f"\nTotal raw rows: {len(raw):,}")

Loaded 201901-bluebikes-tripdata.csv: 69,872 rows
Loaded 201902-bluebikes-tripdata.csv: 80,466 rows


Loaded 201903-bluebikes-tripdata.csv: 102,369 rows


Loaded 201904-bluebikes-tripdata.csv: 166,694 rows


Loaded 201905-bluebikes-tripdata.csv: 223,084 rows


Loaded 201906-bluebikes-tripdata.csv: 274,083 rows


...

Loaded 202309-bluebikes-tripdata.csv: 419,874 rows


Loaded 202310-bluebikes-tripdata.csv: 435,867 rows


Loaded 202311-bluebikes-tripdata.csv: 270,816 rows


Loaded 202312-bluebikes-tripdata.csv: 169,380 rows



Total raw rows: 9,973,245


## Data Cleaning

A few cleaning steps before we build any features:

1. Parse the coordinate columns to numeric values.
2. Compute `tripduration` from timestamps for the new-schema files since they don't include it directly.
3. Drop rows missing essential fields (timestamps, duration, station IDs).
4. Drop unreasonable trip durations (under 1 minute or over 24 hours, which are almost always dock glitches or data errors).
5. Keep only trips that start inside a sensible Boston-area coordinate box.

In [5]:
raw['start_lat'] = pd.to_numeric(raw['start_lat'], errors='coerce')
raw['start_lng'] = pd.to_numeric(raw['start_lng'], errors='coerce')
raw['end_lat'] = pd.to_numeric(raw['end_lat'], errors='coerce')
raw['end_lng'] = pd.to_numeric(raw['end_lng'], errors='coerce')

raw['tripduration'] = pd.to_numeric(raw.get('tripduration'), errors='coerce')
mask_no_dur = raw['tripduration'].isna()
if mask_no_dur.any():
    raw.loc[mask_no_dur, 'tripduration'] = (
        raw.loc[mask_no_dur, 'stoptime'] - raw.loc[mask_no_dur, 'starttime']
    ).dt.total_seconds()
    print(f"Computed tripduration for {mask_no_dur.sum():,} rows from timestamps")

raw = raw.dropna(subset=['starttime', 'tripduration', 'start_station_id', 'end_station_id'])
raw = raw[(raw['tripduration'] >= 60) & (raw['tripduration'] <= 86400)]

raw = raw.dropna(subset=['start_lat', 'start_lng'])
raw = raw[
    (raw['start_lat'] > 42.2) & (raw['start_lat'] < 42.5) &
    (raw['start_lng'] > -71.2) & (raw['start_lng'] < -70.9)
]

print(f"Rows after cleaning: {len(raw):,}")

Computed tripduration for 3,200,875 rows from timestamps


Rows after cleaning: 9,890,715


## Feature Engineering

### Temporal Features

In [6]:
raw['year'] = raw['starttime'].dt.year
raw['month'] = raw['starttime'].dt.month
raw['hour'] = raw['starttime'].dt.hour
raw['day_of_week'] = raw['starttime'].dt.day_name()
raw['date'] = raw['starttime'].dt.date
raw['is_weekend'] = raw['starttime'].dt.dayofweek >= 5
raw['day_type'] = raw['is_weekend'].map({True: 'Weekend', False: 'Weekday'})
raw['trip_minutes'] = (raw['tripduration'] / 60.0).round(1)
raw['period'] = raw['year'].map({
    2019: 'Pre-Pandemic (2019)',
    2022: 'Post-Pandemic (2022)',
    2023: 'Post-Pandemic (2023)'
})

### Demographic Features

Demographics are only available for 2019, since Bluebikes stopped collecting birth year and gender starting in 2022. We kept the code here for completeness but none of the final visualizations on the website use demographic data.

In [7]:
raw['birth_year'] = pd.to_numeric(raw.get('birth_year'), errors='coerce')
raw['gender'] = pd.to_numeric(raw.get('gender'), errors='coerce')
raw['gender_label'] = raw['gender'].map({1: 'Male', 2: 'Female'})
raw['age'] = raw['year'] - raw['birth_year']

raw.loc[(raw['age'] <= 10) | (raw['age'] > 80), 'age'] = None
raw.loc[raw['age'].isna(), 'gender_label'] = None

bins = list(range(15, 85, 5))
labels = [f'{i}-{i+4}' for i in range(15, 80, 5)]
raw['age_group'] = pd.cut(raw['age'], bins=bins, labels=labels, right=False)

print(f"Total cleaned rows: {len(raw):,}")
for yr in YEARS:
    subset = raw[raw['year'] == yr]
    print(f"  {yr}: {len(subset):,} trips, {subset['start_station_id'].nunique()} stations")

Total cleaned rows: 9,890,715


  2019: 2,519,965 trips, 335 stations


  2022: 3,738,722 trips, 433 stations


  2023: 3,632,028 trips, 861 stations


---
## Building Aggregated Datasets

Each visualization gets its own dataset. Every number in these summaries is computed from the full cleaned data, so the counts and averages are exact rather than approximations from a sample.

### Dataset 1: Monthly Ridership by User Type

Built for the Monthly Ridership line chart on the site. Groups by year, month, and user type (Subscriber vs Casual) and counts trips in each bucket. This same dataset also powers the Casual Rider Growth chart by aggregating further up to the annual level.

In [8]:
viz1 = raw.groupby(['year', 'month', 'usertype']).agg(
    trip_count=('tripduration', 'size'),
    avg_duration_min=('trip_minutes', 'mean')
).reset_index()
viz1['avg_duration_min'] = viz1['avg_duration_min'].round(1)
viz1['period'] = viz1['year'].map({
    2019: 'Pre-Pandemic (2019)',
    2022: 'Post-Pandemic (2022)',
    2023: 'Post-Pandemic (2023)'
})
viz1['month_name'] = viz1['month'].map({
    1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
    7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'
})

totals = viz1.groupby(['year', 'month'])['trip_count'].sum().reset_index(name='month_total')
viz1 = viz1.merge(totals, on=['year', 'month'])
viz1['percentage'] = (viz1['trip_count'] / viz1['month_total'] * 100).round(1)

viz1.to_csv(os.path.join(OUTPUT_DIR, 'viz1_ridership.csv'), index=False)
print(f"viz1_ridership.csv: {len(viz1)} rows")

viz1_ridership.csv: 72 rows


### Dataset 2: Activity Heatmap

Built for the heatmap. Counts trips for every (year, day of week, hour) combination, then divides by the number of times that day of the week actually occurred in the year. That normalization matters because otherwise months or years with more weekdays vs weekends would look busier than they actually are.

In [9]:
heatmap_trips = raw.groupby(['year', 'day_of_week', 'hour']).size().reset_index(name='total_trips')
heatmap_days = raw.groupby(['year', 'day_of_week'])['date'].nunique().reset_index(name='num_days')
viz2 = heatmap_trips.merge(heatmap_days, on=['year', 'day_of_week'])
viz2['avg_trips'] = (viz2['total_trips'] / viz2['num_days']).round(1)
viz2['period'] = viz2['year'].map({
    2019: 'Pre-Pandemic (2019)',
    2022: 'Post-Pandemic (2022)',
    2023: 'Post-Pandemic (2023)'
})

viz2.to_csv(os.path.join(OUTPUT_DIR, 'viz2_heatmap.csv'), index=False)
print(f"viz2_heatmap.csv: {len(viz2)} rows")

viz2_heatmap.csv: 504 rows


### Dataset 3: Demographics Pyramid (2019 only)

Built for a population pyramid that we prototyped but did not keep on the final site. The dataset generation stays here so the pipeline is complete in case we want to revisit it.

In [10]:
demo = raw[(raw['year'] == 2019) & raw['gender_label'].notna() & raw['age_group'].notna()].copy()

viz3_parts = []
for dt in ['Weekday', 'Weekend', 'Total']:
    if dt == 'Total':
        subset = demo
    else:
        subset = demo[demo['day_type'] == dt]
    
    grouped = subset.groupby(['age_group', 'gender_label'], observed=True).size().reset_index(name='count')
    total = grouped['count'].sum()
    grouped['percentage'] = (grouped['count'] / total * 100).round(2)
    grouped['day_type'] = dt
    viz3_parts.append(grouped)

viz3 = pd.concat(viz3_parts, ignore_index=True)
viz3['plot_pct'] = viz3.apply(
    lambda row: -row['percentage'] if row['gender_label'] == 'Male' else row['percentage'],
    axis=1
)

viz3.to_csv(os.path.join(OUTPUT_DIR, 'viz3_demographics.csv'), index=False)
print(f"viz3_demographics.csv: {len(viz3)} rows")

viz3_demographics.csv: 78 rows


### Dataset 4: Station Data

Built for the D3 station map. Computes departures and arrivals per station per year, along with total trips and net flow. We keep latitude and longitude at full precision here since the map matches stations across years by rounded coordinates at render time.

In [11]:
starts = raw.groupby(
    ['year', 'start_station_id', 'start_station_name', 'start_lat', 'start_lng']
).size().reset_index(name='departures')
starts.columns = ['year', 'station_id', 'station_name', 'latitude', 'longitude', 'departures']

ends = raw.groupby(
    ['year', 'end_station_id', 'end_station_name', 'end_lat', 'end_lng']
).size().reset_index(name='arrivals')
ends.columns = ['year', 'station_id', 'station_name', 'latitude', 'longitude', 'arrivals']

viz4 = starts.merge(ends, on=['year', 'station_id', 'station_name', 'latitude', 'longitude'], how='outer')
viz4['departures'] = viz4['departures'].fillna(0).astype(int)
viz4['arrivals'] = viz4['arrivals'].fillna(0).astype(int)
viz4['total_trips'] = viz4['departures'] + viz4['arrivals']
viz4['net_flow'] = viz4['arrivals'] - viz4['departures']
viz4['period'] = viz4['year'].map({
    2019: 'Pre-Pandemic (2019)',
    2022: 'Post-Pandemic (2022)',
    2023: 'Post-Pandemic (2023)'
})
viz4 = viz4[viz4['total_trips'] >= 100]

viz4.to_csv(os.path.join(OUTPUT_DIR, 'viz4_stations.csv'), index=False)
print(f"viz4_stations.csv: {len(viz4)} rows")

viz4_stations.csv: 2073 rows


### Dataset 5: Trip Duration Distribution

Built for the duration chart. Bins trip durations into 2-minute intervals and counts trips per (year, day type, user type, bin). We filter to trips under 60 minutes since the long tail past that point is dominated by outliers that would wash out the meaningful differences in the 5 to 20 minute range where most rides sit.

In [12]:
capped = raw[raw['trip_minutes'] <= 60].copy()
dur_bins = list(range(0, 62, 2))
dur_labels = [f'{i}-{i+2}' for i in range(0, 60, 2)]
capped['duration_bin'] = pd.cut(capped['trip_minutes'], bins=dur_bins, labels=dur_labels, right=False)

viz5 = capped.groupby(
    ['year', 'day_type', 'usertype', 'duration_bin'], observed=True
).size().reset_index(name='count')
viz5['period'] = viz5['year'].map({
    2019: 'Pre-Pandemic (2019)',
    2022: 'Post-Pandemic (2022)',
    2023: 'Post-Pandemic (2023)'
})
viz5['duration_bin'] = viz5['duration_bin'].astype(str)
viz5['bin_start'] = viz5['duration_bin'].str.split('-').str[0].astype(int)

viz5.to_csv(os.path.join(OUTPUT_DIR, 'viz5_duration.csv'), index=False)
print(f"viz5_duration.csv: {len(viz5)} rows")

viz5_duration.csv: 360 rows


## Summary

All five datasets are written to the `data/` folder. Verifying that the combined row count comfortably exceeds the 2,000 minimum required by the assignment.

In [13]:
files = ['viz1_ridership.csv', 'viz2_heatmap.csv', 'viz3_demographics.csv', 'viz4_stations.csv', 'viz5_duration.csv']
total_rows = 0
for f in files:
    path = os.path.join(OUTPUT_DIR, f)
    n = len(pd.read_csv(path))
    total_rows += n
    print(f"{f}: {n} rows")

print(f"\nTotal rows across all datasets: {total_rows}"
      f" ({'PASS' if total_rows > 2000 else 'FAIL'}: > 2,000 requirement)")

viz1_ridership.csv: 72 rows
viz2_heatmap.csv: 504 rows
viz3_demographics.csv: 78 rows
viz4_stations.csv: 2073 rows
viz5_duration.csv: 360 rows

Total rows across all datasets: 3087 (PASS: > 2,000 requirement)
